<a href="https://colab.research.google.com/github/starlton/Deep-Learning/blob/main/week2_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — PCA on MNIST

**Goal:** Apply our from-scratch PCA implementation to MNIST (70,000 handwritten digit images, 784 dimensions each) and visualize the results.

**Key questions we'll answer:**
- Can we compress 784 dimensions down to 2 and still see meaningful structure?
- How many components do we need to capture 80% of MNIST's variance?
- Does our PCA match sklearn's implementation?

---

## Setup — Imports

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

%matplotlib inline

---
## 1. Load MNIST

MNIST contains 70,000 grayscale images of handwritten digits (0–9).
Each image is 28×28 pixels, flattened into a vector of **784 features**.

- `X` shape: **(70000, 784)** — 70,000 samples, 784 pixel features each
- `y` shape: **(70000,)** — digit label (0–9) for each image

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1)

In [ ]:
X = mnist['data'].to_numpy()   # Convert from DataFrame to NumPy array

X.shape

---
## 2. PCA From Scratch

Our `my_pca` function implements all 4 steps using only NumPy:

| Step | Operation | Purpose |
|---|---|---|
| 1 | `X - mean` | Remove location bias |
| 2 | `np.cov(X.T)` | Capture feature relationships |
| 3 | `np.linalg.eigh` + sort | Find directions of max variance |
| 4 | `X_centered @ top_evecs` | Project into PCA space |

**Why `eigh` not `eig`?** Covariance matrices are always symmetric, so `eigh` is faster and guaranteed to return real eigenvalues.

**Why sort descending?** `eigh` returns eigenvalues in ascending order. We flip so PC1 (most variance) comes first.

In [ ]:
def my_pca(X, n_components):
    """
    PCA from scratch using only NumPy.

    Steps:
    1. Mean-center X
    2. Compute covariance matrix
    3. Get eigenvectors via eigh, sort descending
    4. Project X onto top n eigenvectors
    """
    # Step 1: Mean-center
    X_centered = X - np.mean(X, axis=0)

    # Step 2: Covariance matrix
    cov = np.cov(X_centered.T)

    # Step 3: Eigenvectors, sorted descending
    evals, evecs = np.linalg.eigh(cov)
    sorted_idx = np.argsort(evals)[::-1]
    evecs = evecs[:, sorted_idx]
    evals = evals[sorted_idx]

    # Step 4: Project
    top_evecs = evecs[:, :n_components]
    X_projected = X_centered @ top_evecs

    # Also return explained variance ratio
    explained_variance_ratio = evals / evals.sum()

    return X_projected, explained_variance_ratio

---
## 3. Project MNIST to 2D

Running PCA on all 70,000 images would require a **(784×784) covariance matrix** and be very slow.
We use the first **10,000 samples** as a representative subset.

Shape check:
- Input: `(10000, 784)`
- Covariance matrix: `(784, 784)`
- Output (n_components=2): `(10000, 2)` ← each image now has 2 coordinates

In [ ]:
X_subset = X[:10000]
X_subset.shape

In [ ]:
X_sub_proj, evr = my_pca(X_subset, 2)

X_sub_proj.shape

---
## 4. Visualize — My PCA in 2D

Each point is one MNIST image projected onto its top 2 principal components.
Points are colored by digit class (0–9). If PCA is working, similar digits should cluster together.

In [ ]:
y_labels = mnist.target[:10000].astype(int)

scatter = plt.scatter(X_sub_proj[:, 0], X_sub_proj[:, 1],
                      c=y_labels,
                      cmap='tab10',
                      s=1,
                      alpha=0.5)
plt.colorbar(scatter, ticks=range(10))
plt.title("2D Projection of MNIST (10,000) samples)")
plt.show()

---
## 5. Verification — My PCA vs sklearn

sklearn's `PCA` uses **SVD** (Singular Value Decomposition) internally rather than eigendecomposition.
Both methods give the same result, but SVD is more numerically stable.

We compare with `np.abs` to handle **sign flips** — eigenvectors can point in either direction
(`[0.7, 0.7]` and `[-0.7, -0.7]` are both valid) so sklearn and our implementation may disagree on sign.

In [ ]:
from sklearn.decomposition import PCA

sklearn_PCA = PCA(n_components=2)
X_sklearn = sklearn_PCA.fit_transform(X_subset)

In [ ]:
y_labels = mnist.target[:10000].astype(int)

scatter = plt.scatter(X_sklearn[:, 0], X_sklearn[:, 1],
                      c=y_labels,
                      cmap='tab10',
                      s=1,
                      alpha=0.5)
plt.colorbar(scatter, ticks=range(10))
plt.title("2D Projection of MNIST (10,000) samples)")
plt.show()

In [ ]:
# Compare absolute values to handle sign flips
match = np.allclose(np.abs(X_sub_proj), np.abs(X_sklearn))
print("\nOutputs match (up to sign flips):", match)

---
## 6. Explained Variance — How Many Components Do We Need?

Each principal component captures a fraction of the total variance in the data.
The **explained variance ratio** tells us what percentage each component contributes.

Key question: **how many components do we need to retain 80% of MNIST's information?**

This tells us the minimum dimensionality we need for a useful compression of the data.

In [ ]:
# Run PCA w/ 50 components to get variance ratios
_, evr_50 = my_pca(X_subset, 50)
evr_50 = evr_50[:50]  # Keep top 50

cumulative = np.cumsum(evr_50)

# Find how many components to hit 80%
n_80 = np.argmax(cumulative >= 0.80) + 1
print(f"Components needed for 80% variance: {n_80}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(1, 51), evr_50 * 100, color='steelblue')
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance (%)")
ax1.set_title("Variance per Component (Top 50)")

ax2.plot(range(1, 51), cumulative * 100, color='steelblue')
ax2.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% threshold')
ax2.axvline(x=n_80, color='orange', linestyle='--', alpha=0.7, label=f'{n_80} components')
ax2.set_xlabel('Number of components')
ax2.set_ylabel('Cumulative Variance (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()

plt.tight_layout()
plt.show()

### Result: How many components capture 80% of MNIST's variance?

**It takes ~43 components to capture 80% of MNIST's variance.**

This means we can compress 784 dimensions down to 43 and still retain 80% of the information — a **18x reduction** in dimensionality. This is exactly why PCA is used for preprocessing before training ML models: fewer dimensions means faster training with minimal information loss.

---
## 7. Observations — What Does the 2D Plot Tell Us?

**Digits that cluster cleanly:**
- **0** and **1** tend to separate well — they have very distinct shapes (circle vs. vertical line)
- **6** and **7** often appear in relatively isolated regions

**Digits that overlap:**
- **4, 7, and 9** frequently overlap — they share similar stroke patterns (vertical lines)
- **3, 5, and 8** often bleed into each other — all have curved right sides

**Why some digits are harder to separate in 2D PCA:**

PCA finds the directions of maximum *global* variance — it has no knowledge of digit labels. It's an unsupervised method. So two digits can overlap in 2D PCA space even if they look visually distinct, because the directions that separate them may not be the top 2 principal components.

Think of it this way: 784 dimensions compressed to 2 inevitably loses information. The first two components capture maybe 15–20% of total variance. The remaining 80%+ — which likely contains the fine details that distinguish similar digits — gets thrown away.

This is why methods like **t-SNE** or **UMAP** (which are non-linear) produce much cleaner digit clusters than PCA — they're designed specifically to preserve local structure, not just global variance.

---

## Week 2 Summary

| Concept | What we learned |
|---|---|
| Eigenvectors | Directions a matrix only scales, never rotates |
| Eigenvalues | The scale factor — how much variance that direction captures |
| Covariance matrix | Describes how features relate to each other; always symmetric |
| PCA | Find eigenvectors of covariance matrix, project data onto top n |
| MNIST result | 43 components capture 80% of variance; 784 → 2 still shows structure |

**Next: Week 3 — Calculus, chain rule, and building micrograd (our own autograd engine) from scratch.**